# TP4, part 1: an LLM through an API (30 min)

Today you build an **agent** and measure it (part 2: the README of this repository, sections 1 to 4). First, the
building block: one call to a language model through an API, a strict output format that a program can read, and a
score on labelled examples.

Run it on your machine, from the tutorial folder: `source .venv/bin/activate`, then `jupyter lab part1-api.ipynb`.
It uses the NVIDIA key you connected in OpenCode (section 0 of the README), or `NVIDIA_API_KEY` if it is set.

In [ ]:
import json
import os
import re
import time
from openai import OpenAI

# The key: NVIDIA_API_KEY if set, else the one you connected in OpenCode (/connect), which OpenCode stores here.
auth = os.path.expanduser("~/.local/share/opencode/auth.json")
if not os.environ.get("NVIDIA_API_KEY") and os.path.exists(auth):
    key = json.load(open(auth)).get("nvidia", {}).get("key")
    if key:
        os.environ["NVIDIA_API_KEY"] = key

# NVIDIA NIM speaks the OpenAI format: only the base URL, the key and the model name change between providers
# (OpenRouter: https://openrouter.ai/api/v1, a local vLLM or Ollama server: http://localhost:.../v1).
if os.environ.get("NVIDIA_API_KEY"):
    client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=os.environ["NVIDIA_API_KEY"])
    MODEL = "nvidia/nemotron-3.5-lightning-30b-a3b"  # the model of part 2: a reasoning model
else:  # the class proxy (its token counts may be missing)
    client = OpenAI(base_url=os.environ["CLASS_API_URL"], api_key=os.environ["CLASS_API_KEY"])
    MODEL = "deepseek-v4-flash-0731"

def ask(prompt, model=MODEL, max_tokens=4096, tries=3):
    """One chat call. Returns (answer, generated tokens, seconds); the answer is "" if every try failed."""
    for attempt in range(tries):
        start = time.time()
        try:
            reply = client.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}],
                                                   max_tokens=max_tokens, timeout=180)
            return reply.choices[0].message.content or "", reply.usage.completion_tokens, time.time() - start
        except Exception as e:  # the free endpoint is shared: wait, then retry
            print("retrying after:", str(e)[:80])
            time.sleep(10 * (attempt + 1))
    return "", 0, 0.0

### 👉 Question 1: Your first call, and what reasoning costs

1. Ask `MODEL` a one-sentence question and print the answer, the number of generated tokens and the time.
2. Ask the multi-step question below. Check the answer by hand. Compare the number of generated tokens with the
   length of the answer (about 1.3 tokens per word): where do the other tokens go, and who pays for them?

In [ ]:
question = ("A train leaves Paris at 2 PM at 100 km/h. A car leaves Lyon, 450 km away, at 3 PM towards Paris at "
            "80 km/h. How far from Paris do they meet?")
#<answer>🔴📝🔴
#</answer>

### 👉 Question 2: An output a program can read

We want a **spelling-error detector**: given a text, list its misspelled words. A program will use the answer, so
we ask for strict JSON: an array of strings, `[]` if there is no error.

**Your task:** write `detection_prompt(text)` (a role, the task, the exact output format, and an example with and
without errors), then complete `detect(text)`, which returns the set of flagged words, or `None` if the answer
contains no valid JSON array.

In [ ]:
def detection_prompt(text):
    #<answer>🔴📝🔴
    #</answer>
    return prompt

def detect(text, max_tokens=4096):
    answer, _, _ = ask(detection_prompt(text), max_tokens=max_tokens)
    flagged = None
    #<answer>🔴📝🔴
    #</answer>
    return flagged

print(detect("I definately need to recieve that package."))

### 👉 Question 3: Measure it

`GOLD` holds 14 sentences and their real errors (six have none). Run `detect` on each (`detect_all`
sends the calls in parallel, which is faster than one after another) and report
**precision** (flagged words that are real errors) and **recall** (real errors that were flagged), counting an invalid
output as flagging nothing. Print every miss and false alarm: what do the misses have in common? Change one thing
in your prompt and measure again: did it help, or was it noise?

In [ ]:
GOLD = [
    ("I think their going to announce the results today.", {"their"}),
    ("The dog wagged it's tail when we arrived.", {"it's"}),
    ("We need to loose two kilos before the race.", {"loose"}),
    ("Please except my apologies for the delay.", {"except"}),
    ("The new policy will effect every employee.", {"effect"}),
    ("She is a remarkably consistant runner.", {"consistant"}),
    ("The comittee publishes its report in Febuary.", {"comittee", "Febuary"}),
    ("The analysis of the survey data was acurate.", {"acurate"}),
    ("The colour of the theatre curtains was grey.", set()),
    ("Nguyen and Siobhan presented their tokenizer at NeurIPS.", set()),
    ("The defenestration of Prague started a long war.", set()),
    ("The judgement was delivered in the autumn.", set()),
    ("We used a regex and a dequeue operation to parse the logs.", set()),
    ("Its main advantage is that it loses no data.", set()),
]
from concurrent.futures import ThreadPoolExecutor
def detect_all(texts):  # detect() on each text, 8 calls at a time; results in the order of the texts
    with ThreadPoolExecutor(8) as pool:
        return list(pool.map(detect, texts))

tp = n_flagged = n_gold = 0
#<answer>🔴📝🔴
#</answer>
print(f"precision {tp / max(n_flagged, 1):.2f}  recall {tp / max(n_gold, 1):.2f}  ({n_gold} errors)")

## From a call to an agent

You now have the pattern of every LLM system: **a prompt, an output a program can read, a labelled set, a score**.
Part 2 keeps the pattern but gives the model tools and lets it decide its next step: an agent, configured in a
file and scored by a script you cannot edit. Go back to the README, section 1.

### 👉 Question 4: A longer text

Hide the 14 sentences among a hundred correct ones and ask for all the errors in one call. How do precision and
recall change as the text grows? (Lecture 4: evaluating the context window.) The model's hidden reasoning grows with
the text: a long input needs `detect(text, max_tokens=16384)`, otherwise the answer is cut off and comes back empty.
Count an empty answer apart from a wrong one.

In [ ]:
#<answer>🔴📝🔴
#</answer>